# Web Scrapping de La Jornada

En este notebook se va a realizar un código capaz de extraer información de la página web (pública) del periódico ***La Jornada*** mediante la técnica conocida como **Web Scrapping** con el objetivo de recabar datos sobre eventos que puedan ser catalogados como *masacres*, los cuales deben de contar con al menos 4 cadáveres u homicidios en el mismo evento.

La extracción de información se hará sobre periódicos del **01 de Enero del 2013** al **31 de Diciembre del 2015** los cuales tienen una periodicidad diaria, salvo en fechas especiales. En la Web se tiene el registro de la Portada y Contraportada del periódico, así como el hipervínculo hacia los artículos y las noticias que aparecen en dicha portada y contraportada. Solo se tomarán en cuenta aquellas noticias y artículos que aparezcan en tales hipervínculos pues no se tiene otra forma de acceder al periódico completo. Esto, en principio, no sesga la información ni merma la obtención de datos pues los datos que se quieren extraer hacen referencia a hechos y eventos de violencia extrema, que tienen un alto impacto en la comunidad, y por lo tanto suelen aparecer en alguna de las portadas.

## Parte 1: Acceso a la página web

Para lograr la recopilación de información, primero se debe acceder de cierta forma a la página web, por lo que se utiliza la librería `request` de python, capaz de enviar peticiones tipo `GET` y `POST`. 

Luego, se usa la librería `bs4` capaz de adaptar y filtrar el formato `HTML` que devuelve la petición anterior y así poder extraer la información en texto plano.


## Parte 2: Extracción de las URL

En el flujo de trabajo, primero se extraen todas las **URL** junto al título de todas las noticias que aparecen en todas las portadas y contraportadas dentro de las fechas establecidas. Luego, se comienza con un proceso de filtrado de notas: debido a que se extraen todas las noticias, se incluyen las deportivas, económicas, de espectáculos, internacionales, entre otras secciones que no resultan de interés para el presente estudio; por lo que se pasa por un proceso de filtrado que solo mantiene aquellas noticias que pertenecen a las secciones:

- Política
- Sociedad
- Estados
- Capital



In [15]:
# =========================================
#        Parte 1 y 2: Extraer URL's
# =========================================


# +++++ CODIGO "LEVE", SOLO ENVÍA UN REQUEST POR PORTADA (TOTAL 1095) +++++


import requests # Peticiones get y post
from bs4 import BeautifulSoup # Manejo de html
from datetime import date, timedelta # Manejo de fechas
import lxml
import re # Expresiones regulares
import time # Para ver cuanto tarda
from tqdm.notebook import tqdm # Barra de carga

t1 = time.time()

url_inicial =  "https://web.jornada.com.mx/" 
# A esta URL hay que agregarle al final AAAA/MM/DD para ver la portada y contraportada de esa fecha

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8',
    'Referer': 'https://www.google.com.mx/'
} # Los encabezados que "disfrazan" el script de python como si fuera un usuario de chrome

lista_urls = [] # Aqui se inicia la variable para guardar los URL's

secciones_de_interes = ['politica',
                        'sociedad',
                        'estados',
                        'capital']


# Hay 365 dias por año (el rango de fechas no incluye biciestos), y se quieren ver 3 años = 1095 días
# Se construye una función que indica la fecha tras sumar x días desde el 01/01/2013:
def fecha(i: int):
    '''Devuelve el string de la fecha que resulta de sumar i días al 2013/01/01 (AAAA/MM/DD)'''
    fecha_inicial = date.fromisoformat("2013-01-01")
    fecha_fin = fecha_inicial + timedelta(days = i)
    return re.sub(r"-","/",str(fecha_fin))


n_dias = 15
n_url = 0 # Contador de chequeo, se puede quitar después
n_url_util = 0 # Contador de chequeo, se puede quitar después     
urls_diarios = [url_inicial]*n_dias # Aquí se va a guardar la lista de todos los URL
for i in range(n_dias):
    urls_diarios[i] = urls_diarios[i] + fecha(i)

pbar = tqdm(urls_diarios, desc = "Extrayendo URL's", unit = "dias")

for url_i in pbar:
    respuesta = requests.get(url_i, headers = headers, timeout = 10) # Hacer petición
    
    # Extrae el HTML
    if str(respuesta.status_code)[0] == '2': # Si el estatus es Exitoso...
        soup = BeautifulSoup(respuesta.text, 'html.parser')
    
        # Primero, traer todas las secciones donde hay noticias (Algunas con hipervinculos)
        lista_divs = soup.find_all("div", class_ = re.compile(r"sumario blanco p16|cabeza|aviso"))
        pbar.set_description(f"Buscando en la fecha: {str(url_i)[27:]}")
        # Extraer todas las URL de los divs
        for div in lista_divs:
            try:
                # Intentar sacar el URL (Algunos no tienen)
                a = div.find('a')
                href = str(a.get('href'))
                section = re.search(r"([^/]+)/", href)
                section = section.group(1)
                n_url += 1
                
                if section in secciones_de_interes: # Filtrar URL's
                    lista_urls.append(url_i + "/" + href )
                    n_url_util += 1
                
                pbar.set_postfix({"URL's totales":n_url,"URL's post-filtro":n_url_util})
                

            except AttributeError:
                continue
        time.sleep(1)

    # Luego, en la sección de abajo donde hay 2 o 3 noticias por sección, se extraen de un XML:
    url_xml = url_i + '/dir.xml'
    respuesta_xml = requests.get(url_xml, headers = headers, timeout = 10)
    if str(respuesta_xml.status_code)[0] == '2': # Si el estatus es Existoso...
        soup_xml = BeautifulSoup(respuesta_xml.content, 'lxml-xml') # Recoger el html
        
        # Primero, traer todas las secciones donde hay noticias (Algunas con hipervinculos)
        lista_items = soup_xml.find_all("Item")
        
        # Extraer las URL de los Items
        for item in lista_items:
            try:
                url_id = str(item.get('id'))
                url_section = str(item.get('section'))
                n_url += 1
                if url_section in secciones_de_interes: # Filtrar URL's
                    lista_urls.append( url_i + "/" + url_section + "/" + url_id )
                    n_url_util += 1
            except Exception as e:
                continue
        time.sleep(.5)
        
print(f"\n\nEl programa tardó: {time.time()-t1:10.4f} segundos")
print(f"\nSe encontraron {n_url:8d} URL's en {n_dias:5d} periódicos, de las cuales tras filtrar por secciones solo se mantienen {n_url_util:6d} URL's")

Extrayendo URL's:   0%|          | 0/15 [00:00<?, ?dias/s]



El programa tardó:    37.8773 segundos

Se encontraron     1886 URL's en    15 periódicos, de las cuales tras filtrar por secciones solo se mantienen   1046 URL's


## Parte 3: Filtrado de información

De las noticias anteriormente extraidas, muchas no hacen referencia a ningun evento criminal o violento, por lo que se realiza un segundo filtrado.
Solo se mantienen las URL tales que en el contenido de la noticia (ya sea en el título, pies de página o los párrafos) contenga alguna de las siguientes ***palabras clave*** o sus variantes (plurales, singulares, masculinos, femeninos, conjugaciones).

- Masacre(s), masacraron, masacrado(s)
- Matanza(s)
- Multihomicidio(s)
- Plurihomicidio(s)
- Exterminio(s), exterminad(os-as), exterminación
- Asesinato(s), asesinad(os-as), asesin(os-as)
- Cadáver(es)
- Homicidio(s), homicida
- Muerte(s), muerto(s), mueren
- Murieron
- Matar, mataron
- Sin vida
- Perdieron la vida
- Agresión, agresores, agredidos, agredió, agredieron
- Balacera
- Tiroteo(s), tirotearon
- Disparó, Dispararon, Disparad(os-as)
- Defunción, defunciones
- Fallecido, fallecida, fallecieron, falleció
- Feminicidio, feminicidios, feminicida
- Torturad(os-as), Tortura(s)
- Maniatad(os-as) = Con manos atadas
- Decapitad(os-as)
- Desmembrad(os-as)
- Ejecucion(es), ejecutó, ejecutad(os-as)
- Abatid(os-as), Abatieron, Abaten, Abatió

Aquellas noticias que tengan alguna de estas palabras, serán guardadas en un archivo de excel (`.xlsx`) antes de seguir analizando la siguiente noticia, esto con el fin de tener un modo de protección en caso de que los *request* sean bloqueados; de esta manera no se perderá la información ya recopilada

In [16]:
# =====================================
#       Parte 2: Buscar KeyWords       
# =====================================


# +++++ CÓDIGO PESADO, ENVÍA UN REQUEST POR *NOTICIA* (TOTAL APROX = 80,000) +++++


import re
import time
import json

t1 = time.time()

def guardar_progreso(noticia):
    with open("noticias_con_match.jsonl", "a", encoding = "utf-8") as file:
        file.write(json.dumps(noticia, ensure_ascii=False) + "\n")

# Definimos las raíces para cubrir plurales y variaciones
# \b al inicio y final asegura que busque palabras completas y no partes de otras
patrones = [
    r"masacr\w+",                  # Masacre(s), masacraron, masacrado(s)
    r"matanz\w+",                  # Matanza(s)
    r"multihomicidio\w*",          # Multihomicidio(s)
    r"plurihomicidio\w*",          # Plurihomicidio(s)
    r"extermin\w+",                # Exterminio(s), exterminad(os-as), exterminación
    r"asesin\w+",                  # Asesinato(s), asesinad(os-as), asesin(os-as)
    r"cad[aá]ver\w*",              # Cadáver(es)
    r"homicid\w+",                 # Homicidio(s), homicida
    r"muer\w+",                    # Muerte(s), muerto(s), mueren
    r"murieron",                   # Palabra exacta
    r"matar\w*",                   # Matar, mataron
    r"sin vida",                   # Frase exacta
    r"perdieron la vida",          # Frase exacta
    r"agres\w+",                   # Agresión, agresores, agredidos, agredió, agredieron
    r"balacera\w*",                # Palabra exacta
    r"tirote\w+",                  # Tiroteo(s), tirotearon
    r"dispar\w+",                  # Disparó, Dispararon, Disparad(os-as)
    r"defunci\w+",                 # Defunción, defunciones
    r"falleci\w+",                 # Fallecido, fallecida, fallecieron, falleció
    r"feminicid\w+",               # Feminicidio, feminicidios, feminicida
    r"tortur\w+",                  # Torturad(os-as), Tortura(s)
    r"maniatad\w+",                # Maniatad(os-as) = Con manos atadas
    r"decapita\w+",                # Decapitad(os-as)
    r"desmembrad\w+",              # Desmembrad(os-as)
    r"\bejecut(?!ivo\b|iva\b)\w+", # Para evitar "Ejecutivo o Ejecutiva"
    r"abati\w+"                    # Abatid(os-as), Abatieron, Abaten, Abatió
    
]

# Unimos todo con el operador OR (|)
regex_final = re.compile("|".join(patrones), re.IGNORECASE)

import requests
from bs4 import BeautifulSoup
import re
from tqdm.notebook import tqdm

# Tu lista de URLs únicas (usa set() para asegurar que no haya repetidas)
urls_unicas = list(set(lista_urls))
n_urls = len(urls_unicas)
noticias_relevantes = []
matchs = 0
pbar = tqdm(urls_unicas, desc="Intentando scraping", unit="url")
url_procesadas = set()

for i, url in enumerate(pbar,1):
    try:
        time.sleep(.75) # Esperamos un poco para no saturar el servidor
        
        # Descarga con timeout para que no se trabe el script
        respuesta = requests.get(url, headers = headers, timeout=10)
        
        # Procesamos el texto completo de una vez
        sopa = BeautifulSoup(respuesta.content, 'html.parser')
        
        # Buscamos en todo el artículo (Ignoramos encabezado, menús, publicidad, etc)
        art = sopa.find("div", id = "article-text")
        
        texto_completo = art.get_text(separator=' ', strip=True)
        
        # Buscamos el match
        match = regex_final.search(texto_completo)
        
        
        if match:
            # Guardamos un diccionario de la noticia
            noticia = {
                'url': url,
                'keyword_detectada': match.group(),
                'texto': texto_completo 
            }
            # Si encontramos algo, guardamos y salimos del loop de esta URL
            # Guardamos el match.group() para saber cual palabra activó la alerta
            noticias_relevantes.append(noticia)

            if url not in url_procesadas:
                url_procesadas.add(url)
                guardar_progreso(noticia)
            
            matchs += 1    
            url_display = str(url)[27:]
    
            # Actualizamos la descripción (línea superior)
            pbar.set_description(f"Intentando {url_display}")
            
            # Actualizamos las estadísticas (a la derecha de la barra)
            pbar.set_postfix({"Matchs": matchs})
            

    except Exception as e:
        print(f"Error en {url}:\t{e}")
        continue

t2 = time.time()
print(f"\nEl programá se ejecutó durante {int(t2-t1)//60:3d} min {(t2-t1)%60:6.3f} seg")

print(f"\nSe Obtuvieron {matchs} noticias con coincidencias de alguna palabra clave, y se guardaron para su posterior análisis")

Intentando scraping:   0%|          | 0/930 [00:00<?, ?url/s]


El programá se ejecutó durante  19 min 30.009 seg

Se Obtuvieron 257 noticias con coincidencias de alguna palabra clave, y se guardaron para su posterior análisis


## Parte 4: Análisis de lenguaje natural

Tras filtrar solo aquellas noticias que tengan coincidencias con alguna de las palabras clave, se requiere poder analizar de manera precisa y eficiente el contenido de la noticia para identificar el número de víctimas, el lugar de ocurrencia, la fecha y alguna otra característica del evento que se desee analizar. Para esto, se utilizará la librería `spaCy` en su módulo de `es_core_news_lg`, el cual es la versión más grande, y por lo tanto, potente y precisa para procesar texto escrito en español y categorizarlo (Ubicación, Personajes, Tiempo, etc.).

Tras este análisis de lenguaje natural (NLP), se guardarán los datos en un archivo para una comprobación humana de una muestra, el guardado se harpa individualmente tras cada noticia. Esta es una técnica de *programación defensiva* y permite una mayor seguridad ante bloqueos de la página web

In [19]:
import spacy
import re
import json
from tqdm.notebook import tqdm


# Definimos las palabras que se quieran asociar al número de victimas
keywords = [r"muert[oa]s?",
            r"víctimas?",
            r"cuerpos?",
            r"occisos?",
            r"fallecid[oa]s?",
            r"cadáver(?:es)?",
            r"restos? humanos?",
            r"ejecutad[oa]s?",
            r"abatid[oa]s?", 
            r"decesos?",
            r"homicidios?",
            r"asesinatos?"]
string_kw = "|".join(keywords) # Aqui se unen en una string separados por "|"

# El número de víctimas podría estar con número o con letras (esta expresion regular solo acepta hasta el número 20 en letras)
numeros = [r"un[oa]?",
          r"dos",
          r"tres",
          r"cuatro",
          r"cinco",
          r"seis",
          r"siete",
          r"ocho",
          r"nueve",
          r"diez",
          r"once",
          r"doce",
          r"trece",
          r"catorce",
          r"quince",
          r"dieciséis",
          r"diecisiete",
          r"dieciocho",
          r"diecinueve",
          r"veinte",
          r"\d+"]
string_num = "|".join(numeros)

texto_a_num = {
    "un"        : 1, "una": 1, "uno": 1,
    "dos"       : 2,
    "tres"      : 3,
    "cuatro"    : 4,
    "cinco"     : 5,    
    "seis"      : 6,
    "siete"     : 7,
    "ocho"      : 8,
    "nueve"     : 9,
    "diez"      : 10,
    "once"      : 11,
    "doce"      : 12,
    "trece"     : 13,
    "catorce"   : 14,
    "quince"    : 15,
    "dieciséis" : 16,
    "diecisiete": 17,
    "dieciocho" : 18,
    "diecinueve": 19,
    "veinte"    : 20
}

# La regex final (usando el puente de 0 a 3 palabras)
patron_victimas = re.compile(rf"\b(?:({string_num})\s+(?:\w+\s+){{0,5}}(?:{keywords})|(?:{keywords})\s+(?:\w+\s+){{0,5}}({string_num}))\b",
                            flags = re.IGNORECASE | re.UNICODE)

def filtrar_años(lista):
    for i,item in enumerate(lista):
        if item > 100:
            lista.pop(i)
    return lista

# Cargamos el modelo
nlp = spacy.load("es_core_news_lg")

# COMPARAR REGEX vs spaCy
def procesar_num_victimas(noticia : str):
    doc = nlp(noticia)

    # Extraer ubicaciones (extraes entidades=cateogiras siempre que la etiqueta sea CARDINAL)
    vict_nlp = [ent.text for ent in doc.ents if ent.label_ == "CARDINAL"]


    victimas_nlp = []
    for x in vict_nlp:
        # El Match podría ser el texto "siete" o el número 4. Pasamos todo a int
        if x.isdigit():
            victimas_nlp.append(int(x))
        else:
            victimas_nlp.append(texto_a_num.get(x, 1))

    
    match_victimas = re.search(patron_victimas, noticia)

    if match_victimas:
        # Guardamos el match (el número de victimas)
        victimas_re = match_victimas.group(1) or match_victimas.group(2)
        victimas_re = victimas_re.lower().strip()
        
        # El Match podría ser el texto "siete" o el número 4. Pasamos todo a int
        if victimas_re.isdigit():
            victimas_re = int(victimas_re)
        else:
            victimas_re = texto_a_num.get(victimas_re, 1)

        # CASO DE ALTA CONFIANZA: Ambos sensores coinciden
        if victimas_re in victimas_nlp:
            resultado_final = victimas_re
            nivel_confianza = 3
    
        # CASO DE CONFLICTO: Regex detectó algo pero spaCy no lo confirmó
        else:
            # Priorizamos Regex porque está anclada a una palabra clave (muerto, occiso)
            resultado_final = victimas_re
            nivel_confianza = 2
    else: # Si victimas_re no es > 0 (no hay match)
        # CASO DE RESCATE: Regex no vió nada, pero spaCy detectó números
        if len(victimas_nlp) > 0:
            # Tomamos el primer cardinal que no sea un año (filtro simple > 100)
            candidato = filtrar_años(victimas_nlp)
            if candidato:
                resultado_final = max(candidato)
                nivel_confianza = 1
            else:
                resultado_final = 0
                nivel_confianza = 0
        else:
            resultado_final = 0
            nivel_confianza = 0
            
    return {"num_Victimas": resultado_final, "conf_num_victimas":nivel_confianza}
        


def procesar_loc(texto_limpio: str):
    doc = nlp(texto_limpio)
    
    # --- Extracción de Ubicaciones ---
    # Filtramos solo las entidades etiquetadas como 'LOC' (Ubicaciones)
    ubicaciones = [ent.text for ent in doc.ents if ent.label_ == "LOC"]
    
    # --- Extracción de Víctimas ---
    # Aqui es híbrido entre NLP y reg exp
    # Buscamos números seguidos de palabras clave de victimización
    victimas = 0
    # Patrón: un número (dígito o palabra) seguido de palabras de violencia
    patron_victimas = r"(un[ao]?|dos|tres|cuatro|cinco|seis|siete|ocho|nueve|diez|\d+)\s+(muertos?|víctimas?|fallecidos?|occisos?|cadáveres?|cuerpos?)"
    
    match_victimas = re.search(patron_victimas, texto_limpio, re.IGNORECASE)
    if match_victimas:
        victimas = match_victimas.group(1) # Extrae el número (ej: "cinco" o "5")

    return { "locs_nlp": ubicaciones }



def guardar_progreso_nlp(nlp: dict):
    with open("noticias_nlp.jsonl", "a", encoding = "utf-8") as file:
        file.write(json.dumps(nlp, ensure_ascii=False) + "\n")

        
with open("noticias_con_match.jsonl", "r", encoding = "utf-8") as f:
    contador = sum(1 for linea in f)


pbar = tqdm(total = contador, desc="Procesando texto de noticia", unit="noticias")
        
with open("noticias_con_match.jsonl", "r") as f:
    for linea in f:
        linea_py = json.loads(linea)
        pbar.update(1)
        locs = procesar_loc(linea_py["texto"])
        n_vict = procesar_num_victimas(linea_py["texto"])
        resultado_nlp = linea_py | locs | n_vict
        guardar_progreso_nlp(resultado_nlp)

Procesando texto de noticia:   0%|          | 0/521 [00:00<?, ?noticias/s]

UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 3175: character maps to <undefined>